ESERCIZIO

Implementa una funzione python semplificata per calcolare la distanza di Frechet tra due set di vettori latenti monodimensionali.
Utilizza dati sentiteci generati con medie e varianza differenti per osservare come il punteggio aumenti al crescere della divergenza tra i gruppi.
Suggerimento: usa i vettori medi e le deviazoini stanrda per calcolare la differenza quadratica e la traccia.


In [ ]:
import os

# Configurazione backend
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import ops
import numpy as np
import matplotlib.pyplot as plt

# 1. FUNZIONE DISTANZA DI FRÉCHET SEMPLIFICATA
def calculate_1d_frechet(real_latent, gen_latent):
    """
    Calcola la distanza di Fréchet per vettori 1D.
    Formula: d^2 = (mu1 - mu2)^2 + (sigma1 - sigma2)^2
    """
    # Calcolo medie e deviazioni standard usando Keras Ops (Backend agnostic)
    mu_r = ops.mean(real_latent)
    mu_g = ops.mean(gen_latent)
    
    sigma_r = ops.std(real_latent)
    sigma_g = ops.std(gen_latent)
    
    # Calcolo della distanza quadratica
    diff_mu_sq = ops.square(mu_r - mu_g)
    diff_sigma_sq = ops.square(sigma_r - sigma_g)
    
    fid_score = diff_mu_sq + diff_sigma_sq
    return ops.convert_to_numpy(fid_score)

# 2. GENERAZIONE DATI SINTETICI
# Normale standard: media 0, dev.st. 1
n_samples = 1000
real_data = np.random.normal(0, 1, (n_samples, 1)).astype("float32")

# Simuliamo 3 scenari di generazione con divergenza crescente
scenarios = [
    {"mu": 0.2,  "sigma": 1.1, "label": "Quasi Identico"},
    {"mu": 1.5,  "sigma": 0.5, "label": "Media Spostata"},
    {"mu": 4.0,  "sigma": 2.5, "label": "Molto Divergente"}
]

# 3. TEST E VISUALIZZAZIONE
plt.figure(figsize=(12, 6))
plt.hist(real_data, bins=30, alpha=0.3, label="Dati REALI", color="blue", density=True)

for sc in scenarios:
    # Generiamo i campioni sintetici
    gen_data = np.random.normal(sc["mu"], sc["sigma"], (n_samples, 1)).astype("float32")
    
    # Calcoliamo il punteggio
    score = calculate_1d_frechet(real_data, gen_data)
    
    # Plot della distribuzione generata
    plt.hist(gen_data, bins=30, alpha=0.5, label=f"{sc['label']} (FID: {score:.2f})", density=True)

plt.title("Visualizzazione FID: All'aumentare della divergenza, il punteggio esplode")
plt.xlabel("Valore Latente")
plt.ylabel("Densità")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print("Osservazione: Noterai che più la 'campana' generata si sposta o si allarga rispetto a quella blu, più il FID aumenta.")